In [1]:
import numpy as np

In [ ]:
# import pandas as pd
# import numpy as np



# attack_states = ["Normal", "MiTM", "DoS", "Physical"]

# def prior_from_labels(labels, states=attack_states):
#     counts = labels.value_counts().to_dict()
#     p = np.array([counts.get(s, 0) for s in states], dtype=float)
#     p = p / p.sum()
#     return p, counts

# def label_from_table3(times, table3_df, subsystem_id):
#     labels = pd.Series(["Normal"] * len(times), index=times.index)

#     w = table3_df[table3_df["subsystems"].apply(lambda s: subsystem_id in s)].copy()
#     if w.empty:
#         return labels

#     # precedence so Physical overwrites DoS overwrites MiTM if overlap ever occurs
#     for atk in ["MiTM", "DoS", "Physical"]:
#         ww = w[w["attack_state"] == atk]
#         for _, row in ww.iterrows():
#             mask = (times >= row["start"]) & (times <= row["end"])
#             labels.loc[mask] = atk

#     return labels

# df = pd.read_csv(CSV_PATH)

# # adjust this if your timestamp column name differs
# time_col = "Time"
# df[time_col] = pd.to_datetime(df[time_col], errors="coerce", dayfirst=True)
# df = df.dropna(subset=[time_col]).sort_values(time_col).reset_index(drop=True)
# times = df[time_col]

# # Table 3 events for acquisition 4 taken from your screenshot
# # subsystems are based on the "Affected subsystem" column in Table 3
# table3 = pd.DataFrame([
#     # A1
#     dict(start="2022-02-21 14:47:34", end="2022-02-21 14:47:34", attack_state="MiTM", subsystems={1}),
#     # A6
#     dict(start="2022-02-21 14:52:18", end="2022-02-21 14:52:59", attack_state="DoS", subsystems={1}),
#     # A7
#     dict(start="2022-02-21 14:55:46", end="2022-02-21 14:56:23", attack_state="Physical", subsystems={1}),
#     # A8
#     dict(start="2022-02-21 14:57:26", end="2022-02-21 14:57:46", attack_state="DoS", subsystems={1}),
#     # A9
#     dict(start="2022-02-21 14:58:24", end="2022-02-21 14:59:19", attack_state="Physical", subsystems={1}),
#     # A10
#     dict(start="2022-02-21 15:01:56", end="2022-02-21 15:02:34", attack_state="Physical", subsystems={1}),
#     # A14 affects all subsystems including S1
#     dict(start="2022-02-21 15:12:25", end="2022-02-21 15:13:55", attack_state="DoS", subsystems={1,2,3,4}),
# ])

# table3["start"] = pd.to_datetime(table3["start"])
# table3["end"] = pd.to_datetime(table3["end"])

# labels_s1 = label_from_table3(times, table3, subsystem_id=1)
# p_attack1, counts_attack1 = prior_from_labels(labels_s1)

# print("Counts S1", counts_attack1)
# print("P(AttackType_1) order Normal MiTM DoS Physical", p_attack1)

# print("Number of non Normal samples for S1", int((labels_s1 != "Normal").sum()), "out of", len(labels_s1))


Counts S1 {'Normal': 1430, 'DoS': 153, 'Physical': 133, 'MiTM': 1}
P(AttackType_1) order Normal MiTM DoS Physical [8.32847991e-01 5.82411182e-04 8.91089109e-02 7.74606872e-02]
Number of non Normal samples for S1 287 out of 1717


In [ ]:
# import pandas as pd
# import numpy as np
# import re

# # -----------------------------
# # Paths
# # # -----------------------------
# # CSV_PATH = "/mnt/data/phy_att_4.csv"
# # README_XLSX_PATH = "/mnt/data/README.xlsx"

# attack_states = ["Normal", "MiTM", "DoS", "Physical"]

# # -----------------------------
# # Subsystem definitions (locked)
# # -----------------------------
# S1 = {
#     "tanks": set(range(1, 6)),      # 1..5
#     "pumps": set(range(1, 5)),      # 1..4
#     "valves": set(range(1, 20)),    # 1..19
#     "flows": set()                 # none specified for S1
# }
# S2 = {
#     "tanks": {6},
#     "pumps": set(),
#     "valves": {20},
#     "flows": {1}                   # Flow sensor 1
# }

# # -----------------------------
# # Helpers
# # -----------------------------
# def _find_col(df, candidates):
#     cols_lower = {c.lower(): c for c in df.columns}
#     for cand in candidates:
#         if cand.lower() in cols_lower:
#             return cols_lower[cand.lower()]
#     # partial match fallback
#     for c in df.columns:
#         cl = c.lower()
#         for cand in candidates:
#             if cand.lower() in cl:
#                 return c
#     raise KeyError(f"Could not find any of columns like {candidates}. Found {list(df.columns)}")

# def parse_attack_state(type_text):
#     s = str(type_text).lower()

#     # physical fault
#     if "physical fault" in s or "breakdown" in s or "water leak" in s or "leak" in s:
#         return "Physical"

#     # DoS family
#     if "flood" in s or "dos" in s or "denial" in s or "icmp" in s or "tcp flood" in s or "udp flood" in s:
#         return "DoS"

#     # MiTM bucket includes MITM and scans
#     if "mitm" in s or "man in the middle" in s or "scan" in s or "syn scan" in s or "fin scan" in s or "xmas" in s or "null scan" in s:
#         return "MiTM"

#     # default
#     return "MiTM"

# def extract_assets(effect_text):
#     """
#     From Effect on physical process, extract mentioned tank/pump/valve/flow sensor indices.
#     """
#     s = str(effect_text)

#     tanks = set(int(x) for x in re.findall(r"\btank\s*([0-9]+)\b", s, flags=re.IGNORECASE))
#     pumps = set(int(x) for x in re.findall(r"\bpump\s*([0-9]+)\b", s, flags=re.IGNORECASE))
#     valves = set(int(x) for x in re.findall(r"\bvalve\s*([0-9]+)\b", s, flags=re.IGNORECASE))

#     # flow sensor 1, flow sensor 2 style
#     flows = set(int(x) for x in re.findall(r"\bflow\s*sensor\s*([0-9]+)\b", s, flags=re.IGNORECASE))

#     return tanks, pumps, valves, flows

# import re

# def extract_plcs(type_text):
#     s = str(type_text)
#     hits = re.findall(r"PLC\s*([1-4])", s, flags=re.IGNORECASE)
#     return set(int(h) for h in hits)

# def infer_subsystems(type_text, effect_phys):
#     """
#     Correct rule

#     - Cyber attacks are mapped by the PLC mentioned in 'Type of attack'
#       even when effect_phys says 'No effect on physical process'

#     - Physical faults are mapped by assets mentioned in effect_phys
#       using your subsystem definitions

#     - If effect_phys mentions valve or tank or flow sensor, we also map via assets
#       because sometimes the PLC mentioned is not the physically affected subsystem
#     """
#     affected = set()

#     # 1) PLC mapping from Type of attack
#     plcs = extract_plcs(type_text)
#     for plc in plcs:
#         affected.add(plc)   # PLC1->S1, PLC2->S2, etc

#     # 2) Asset mapping from Effect on physical process
#     tanks, pumps, valves, flows = extract_assets(effect_phys)

#     if (tanks & S1["tanks"]) or (pumps & S1["pumps"]) or (valves & S1["valves"]) or (flows & S1["flows"]):
#         affected.add(1)
#     if (tanks & S2["tanks"]) or (pumps & S2["pumps"]) or (valves & S2["valves"]) or (flows & S2["flows"]):
#         affected.add(2)

#     return affected


# def prior_from_labels(labels, states=attack_states):
#     counts = labels.value_counts().to_dict()
#     p = np.array([counts.get(s, 0) for s in states], dtype=float)
#     p = p / p.sum()
#     return p, counts

# def label_from_windows(times, windows_df, subsystem_id):
#     """
#     times is a pandas Series of timestamps length N
#     windows_df has columns start, end, attack_state, subsystems
#     """
#     labels = pd.Series(["Normal"] * len(times), index=times.index)

#     w = windows_df[windows_df["subsystems"].apply(lambda s: subsystem_id in s)].copy()
#     if w.empty:
#         return labels

#     # precedence
#     for atk in ["MiTM", "DoS", "Physical"]:
#         ww = w[w["attack_state"] == atk]
#         for _, row in ww.iterrows():
#             mask = (times >= row["start"]) & (times <= row["end"])
#             labels.loc[mask] = atk

#     return labels

# def load_readme_attack_table(readme_path):
#     """
#     Load the acquisition 4 attack table from README.xlsx.
#     This tries to find a sheet whose name contains 'attack' and '4'.
#     """
#     xls = pd.ExcelFile(readme_path)
#     sheet = None
#     for sh in xls.sheet_names:
#         sh_l = sh.lower()
#         if "attack" in sh_l and "4" in sh_l:
#             sheet = sh
#             break

#     # fallback if naming is different
#     if sheet is None:
#         # pick the first sheet containing 'attack'
#         for sh in xls.sheet_names:
#             if "attack" in sh.lower():
#                 sheet = sh
#                 break

#     if sheet is None:
#         raise ValueError(f"Could not find an attack sheet in {readme_path}. Sheets: {xls.sheet_names}")

#     df = xls.parse(sheet)

#     start_col = _find_col(df, ["Start time", "Start"])
#     end_col = _find_col(df, ["End time", "End"])
#     type_col = _find_col(df, ["Type of attack", "Attack"])
#     eff_col = _find_col(df, ["Effect on physical process", "Effect on physical"])

#     out = df[[start_col, end_col, type_col, eff_col]].copy()
#     out.columns = ["start", "end", "type_text", "effect_phys"]

#     out["start"] = pd.to_datetime(out["start"], errors="coerce", dayfirst=True)
#     out["end"] = pd.to_datetime(out["end"], errors="coerce", dayfirst=True)
#     out = out.dropna(subset=["start", "end"]).copy()

#     out["attack_state"] = out["type_text"].apply(parse_attack_state)
#     out["subsystems"] = out.apply(lambda r: infer_subsystems(r["type_text"], r["effect_phys"]), axis=1)

#     # keep only rows that map to at least one subsystem
#     out = out[out["subsystems"].apply(lambda s: len(s) > 0)].copy()

#     return out, sheet

# # -----------------------------
# # MAIN: Set 1
# # -----------------------------
# df = pd.read_csv(CSV_PATH)

# time_col = _find_col(df, ["Time", "Timestamp", "DateTime"])
# df[time_col] = pd.to_datetime(df[time_col], errors="coerce", dayfirst=True)
# df = df.dropna(subset=[time_col]).sort_values(time_col).reset_index(drop=True)
# times = df[time_col]

# windows, used_sheet = load_readme_attack_table(README_XLSX_PATH)

# # restrict windows to CSV time span only, so you count Normal properly
# t0, t1 = times.iloc[0], times.iloc[-1]
# windows_in_span = windows[(windows["end"] >= t0) & (windows["start"] <= t1)].copy()

# labels_s1 = label_from_windows(times, windows_in_span, subsystem_id=1)
# p_attack1, counts_attack1 = prior_from_labels(labels_s1)

# print("README sheet used:", used_sheet)
# print("CSV time range:", t0, "to", t1)
# print("Counts S1:", counts_attack1)
# print("P(AttackType_1) [Normal, MiTM, DoS, Physical]:", p_attack1)
# print("Non-Normal seconds for S1:", int((labels_s1 != "Normal").sum()), "out of", len(labels_s1))

# # optional debug: show the windows that actually affected subsystem 1 in this span
# dbg = windows_in_span[windows_in_span["subsystems"].apply(lambda s: 1 in s)][
#     ["start", "end", "attack_state", "type_text", "effect_phys", "subsystems"]
# ]
# print("\nWindows affecting Subsystem 1 within CSV span:")
# print(dbg.to_string(index=False))


README sheet used: attack_4
CSV time range: 2022-02-21 14:45:18 to 2022-02-21 15:13:54
Counts S1: {'Normal': 1430, 'DoS': 153, 'Physical': 133, 'MiTM': 1}
P(AttackType_1) [Normal, MiTM, DoS, Physical]: [8.32847991e-01 5.82411182e-04 8.91089109e-02 7.74606872e-02]
Non-Normal seconds for S1: 287 out of 1717

Windows affecting Subsystem 1 within CSV span:
              start                 end attack_state                                            type_text                                                           effect_phys subsystems
2022-02-21 14:47:34 2022-02-21 14:47:34         MiTM                                SYN scan against PLC1                                         No effect on physical process        {1}
2022-02-21 14:52:18 2022-02-21 14:52:59          DoS ICMP flood against PLC1 from spoofed PLC4 IP address   Delay in opening valve 20: tank 6 start to fill up later than norma  {1, 2, 4}
2022-02-21 14:55:46 2022-02-21 14:56:23     Physical                 physical fault:

In [ ]:
# import pandas as pd
# import numpy as np
# import re



# attack_states = ["Normal", "MiTM", "DoS", "Physical"]
# state_to_idx = {s:i for i,s in enumerate(attack_states)}

# # ----------------------------
# # Helpers
# # ----------------------------
# def _find_col(df, candidates):
#     cols_lower = {c.lower(): c for c in df.columns}
#     for cand in candidates:
#         if cand.lower() in cols_lower:
#             return cols_lower[cand.lower()]
#     for c in df.columns:
#         cl = c.lower()
#         for cand in candidates:
#             if cand.lower() in cl:
#                 return c
#     raise KeyError(f"Could not find columns like {candidates}. Found: {list(df.columns)}")

# def parse_attack_state(type_text):
#     """
#     Map README 'Type of attack' to your 4-state AttackType.
#     Physical is only when it is explicitly a physical fault in the type text.
#     """
#     s = str(type_text).lower()

#     # Physical faults explicitly
#     if "physical fault" in s or "breakdown" in s or "water leak" in s:
#         return "Physical"

#     # DoS style floods
#     if "flood" in s or "dos" in s or "denial" in s:
#         return "DoS"

#     # MiTM bucket includes scans and MITM
#     if "mitm" in s or "man in the middle" in s or "scan" in s:
#         return "MiTM"

#     return "Normal"

# def extract_plcs(text):
#     s = str(text)
#     hits = re.findall(r"PLC\s*([1-4])", s, flags=re.IGNORECASE)
#     return set(int(h) for h in hits)

# def extract_tanks(text):
#     s = str(text).upper()
#     hits = re.findall(r"\bTANK\s*([0-9]+)\b", s)
#     # also catch "tank 6" without word boundary issues
#     hits2 = re.findall(r"\btank\s*([0-9]+)\b", str(text).lower())
#     vals = set(int(h) for h in hits) | set(int(h) for h in hits2)
#     return vals

# def extract_pumps(text):
#     s = str(text).lower()
#     hits = re.findall(r"\bpump\s*([0-9]+)\b", s)
#     return set(int(h) for h in hits)

# def subsystems_for_row(type_text, effect_phys_text):
#     """
#     Returns set of subsystems impacted by THIS ROW.
#     Priority:
#       - If cyber: use PLC numbers
#       - If physical fault: use tank/pump IDs found in type and effect columns
#     """
#     atk = parse_attack_state(type_text)
#     subs = set()

#     if atk in ["MiTM", "DoS"]:
#         plcs = extract_plcs(type_text)
#         # PLC1->S1, PLC2->S2, etc
#         subs |= set(plcs)
#         return subs, atk

#     if atk == "Physical":
#         # Use both fields to find asset IDs
#         tanks = extract_tanks(type_text) | extract_tanks(effect_phys_text)
#         pumps = extract_pumps(type_text) | extract_pumps(effect_phys_text)

#         # Your mapping rules
#         if any(t in {1,2,3,4,5} for t in tanks):  # S1 tanks
#             subs.add(1)
#         if any(p in {1,2,3,4} for p in pumps):    # S1 pumps
#             subs.add(1)
#         if 6 in tanks:                            # S2 tank
#             subs.add(2)

#         return subs, atk

#     return subs, atk

# def load_attack_table(readme_xlsx):
#     xls = pd.ExcelFile(readme_xlsx)

#     # pick all sheets whose name starts with "attack"
#     sheets = [s for s in xls.sheet_names if s.lower().startswith("attack")]
#     if not sheets:
#         sheets = [s for s in xls.sheet_names if "attack" in s.lower()]
#     if not sheets:
#         raise ValueError(f"No attack sheets found. Sheets: {xls.sheet_names}")

#     frames = []
#     for sh in sheets:
#         df = xls.parse(sh)
#         if df.empty:
#             continue

#         start_col = _find_col(df, ["Start time", "Start", "StartTime"])
#         end_col   = _find_col(df, ["End time", "End", "EndTime"])
#         type_col  = _find_col(df, ["Type of attack", "Attack type", "Attack", "Type"])
#         effect_col = _find_col(df, ["Effect on physical process", "Effect on physical", "Physical process"])

#         out = df[[start_col, end_col, type_col, effect_col]].copy()
#         out.columns = ["start", "end", "type_text", "effect_phys"]

#         out["start"] = pd.to_datetime(out["start"], errors="coerce", dayfirst=True)
#         out["end"]   = pd.to_datetime(out["end"], errors="coerce", dayfirst=True)
#         out = out.dropna(subset=["start", "end"])
#         out = out[out["end"] >= out["start"]].copy()

#         # compute subsystems and attack state from row text
#         subs_list = []
#         atk_list = []
#         for _, r in out.iterrows():
#             subs, atk = subsystems_for_row(r["type_text"], r["effect_phys"])
#             subs_list.append(subs)
#             atk_list.append(atk)

#         out["subsystems"] = subs_list
#         out["attack_state"] = atk_list

#         # keep only rows that affect some subsystem OR are explicit physical faults we could map
#         out = out[out["subsystems"].apply(lambda s: len(s) > 0)].copy()

#         frames.append(out)

#     return pd.concat(frames, ignore_index=True)

# def build_attack_series(times, attack_df, subsystem_id):
#     """
#     Label AttackType_s(t) for a subsystem.
#     precedence: Physical > DoS > MiTM > Normal
#     """
#     labels = pd.Series(["Normal"] * len(times), index=times.index)

#     w = attack_df[attack_df["subsystems"].apply(lambda s: subsystem_id in s)].copy()
#     if w.empty:
#         return labels

#     # overwrite in increasing precedence order
#     for atk in ["MiTM", "DoS", "Physical"]:
#         ww = w[w["attack_state"] == atk]
#         for _, row in ww.iterrows():
#             mask = (times >= row["start"]) & (times <= row["end"])
#             labels.loc[mask] = atk

#     return labels

# def conditional_cpt(child_labels, parent_labels, alpha=0.5):
#     """
#     Returns (counts, cpt) where
#       counts shape [child, parent]
#       cpt    shape [child, parent] columns sum to 1
#     alpha is Dirichlet smoothing per cell.
#     """
#     counts = np.zeros((4,4), dtype=int)
#     for c, p in zip(child_labels, parent_labels):
#         counts[state_to_idx[c], state_to_idx[p]] += 1

#     counts_sm = counts.astype(float) + alpha
#     col_sums = counts_sm.sum(axis=0, keepdims=True)
#     cpt = counts_sm / col_sums
#     return counts, cpt

# # ----------------------------
# # MAIN
# # ----------------------------
# df = pd.read_csv(CSV_PATH)
# time_col = _find_col(df, ["Time", "Timestamp", "DateTime"])
# df[time_col] = pd.to_datetime(df[time_col], errors="coerce", dayfirst=True)
# df = df.dropna(subset=[time_col]).sort_values(time_col).reset_index(drop=True)
# times = df[time_col]

# attack_df = load_attack_table(README_XLSX_PATH)

# # Build time-aligned labels for S1 and S2
# A1 = build_attack_series(times, attack_df, subsystem_id=1)
# A2 = build_attack_series(times, attack_df, subsystem_id=2)

# # Compute P(A2 | A1)
# counts_21, cpt_21 = conditional_cpt(A2.values, A1.values, alpha=0.5)

# print("Time range:", times.iloc[0], "to", times.iloc[-1])
# print("\nCounts for AttackType_1:", A1.value_counts().to_dict())
# print("Counts for AttackType_2:", A2.value_counts().to_dict())

# print("\nRaw joint counts matrix counts[AttackType_2, AttackType_1]")
# print("Rows:   ", attack_states)
# print("Cols:   ", attack_states)
# print(counts_21)

# print("\nCPT  P(AttackType_2 | AttackType_1)  columns sum to 1  (alpha=0.5 smoothing)")
# cpt_df = pd.DataFrame(cpt_21, index=[f"A2={s}" for s in attack_states],
#                       columns=[f"A1={s}" for s in attack_states])
# print(cpt_df.round(4))

# # optional sanity: show the windows that affected S2 in this CSV slice
# csv_start, csv_end = times.iloc[0], times.iloc[-1]
# w2 = attack_df[attack_df["subsystems"].apply(lambda s: 2 in s)].copy()
# w2 = w2[(w2["start"] <= csv_end) & (w2["end"] >= csv_start)].sort_values("start")
# print("\nWindows affecting Subsystem 2 within this CSV range:")
# print(w2[["start","end","attack_state","type_text","effect_phys","subsystems"]])


Time range: 2022-02-21 14:45:18 to 2022-02-21 15:13:54

Counts for AttackType_1: {'Normal': 1430, 'DoS': 153, 'Physical': 133, 'MiTM': 1}
Counts for AttackType_2: {'Normal': 1714, 'MiTM': 3}

Raw joint counts matrix counts[AttackType_2, AttackType_1]
Rows:    ['Normal', 'MiTM', 'DoS', 'Physical']
Cols:    ['Normal', 'MiTM', 'DoS', 'Physical']
[[1427    1  153  133]
 [   3    0    0    0]
 [   0    0    0    0]
 [   0    0    0    0]]

CPT  P(AttackType_2 | AttackType_1)  columns sum to 1  (alpha=0.5 smoothing)
             A1=Normal  A1=MiTM  A1=DoS  A1=Physical
A2=Normal       0.9969   0.5000  0.9903       0.9889
A2=MiTM         0.0024   0.1667  0.0032       0.0037
A2=DoS          0.0003   0.1667  0.0032       0.0037
A2=Physical     0.0003   0.1667  0.0032       0.0037

Windows affecting Subsystem 2 within this CSV range:
                 start                 end attack_state  \
28 2022-02-21 14:48:17 2022-02-21 14:48:17         MiTM   
31 2022-02-21 14:51:34 2022-02-21 14:51:34     

### P(Attack Type 1)

In [50]:
import pandas as pd

def read_csv_robust(path):
    # 1) Try the normal way first
    try:
        df = pd.read_csv(path)
        if df.shape[1] > 1:
            return df
    except Exception:
        pass

    # 2) If normal read fails or returns 1 column, try common encodings
    encodings = ["utf-8-sig", "utf-8", "utf-16", "utf-16-le", "utf-16-be", "latin1", "cp1252"]

    last_err = None
    for enc in encodings:
        try:
            tmp = pd.read_csv(path, encoding=enc, nrows=5)

            # If still 1 column, detect delimiter from header string
            if tmp.shape[1] == 1:
                header = tmp.columns[0]
                if "\t" in header:
                    df = pd.read_csv(path, encoding=enc, sep="\t")
                    if df.shape[1] > 1:
                        return df
                if ";" in header:
                    df = pd.read_csv(path, encoding=enc, sep=";")
                    if df.shape[1] > 1:
                        return df

            # Otherwise it parsed fine
            df = pd.read_csv(path, encoding=enc)
            if df.shape[1] > 1:
                return df

        except UnicodeDecodeError as e:
            last_err = e
            continue
        except Exception:
            continue

    raise RuntimeError(f"Could not read {path}. Last UnicodeDecodeError: {last_err}")


In [46]:
import pandas as pd
import numpy as np
import re

README_XLSX_PATH = "README.xlsx"

attack_states = ["Normal", "MiTM", "DoS", "Physical"]

# -----------------------------
# Subsystem definitions (locked)
# -----------------------------
S1 = {
    "tanks": set(range(1, 6)),      # 1..5
    "pumps": set(range(1, 5)),      # 1..4
    "valves": set(range(1, 20)),    # 1..19
    "flows": set()
}
S2 = {
    "tanks": {6},
    "pumps": set(),
    "valves": {20},
    "flows": {1}                   # Flow sensor 1
}

# -----------------------------
# Helpers
# -----------------------------
def _find_col(df, candidates):
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    for c in df.columns:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    raise KeyError(f"Could not find any of columns like {candidates}. Found {list(df.columns)}")

def parse_attack_state(type_text):
    s = str(type_text).lower()

    if "physical fault" in s or "breakdown" in s or "water leak" in s or "leak" in s:
        return "Physical"

    if "flood" in s or "dos" in s or "denial" in s or "icmp" in s or "tcp flood" in s or "udp flood" in s:
        return "DoS"

    if "mitm" in s or "man in the middle" in s or "scan" in s or "syn scan" in s or "fin scan" in s or "xmas" in s or "null scan" in s:
        return "MiTM"

    # conservative default
    return "MiTM"

def extract_plcs(type_text):
    s = str(type_text)
    hits = re.findall(r"PLC\s*([1-4])", s, flags=re.IGNORECASE)
    return set(int(h) for h in hits)

def extract_assets(effect_text):
    s = str(effect_text)

    tanks = set(int(x) for x in re.findall(r"\btank\s*([0-9]+)\b", s, flags=re.IGNORECASE))
    pumps = set(int(x) for x in re.findall(r"\bpump\s*([0-9]+)\b", s, flags=re.IGNORECASE))
    valves = set(int(x) for x in re.findall(r"\bvalve\s*([0-9]+)\b", s, flags=re.IGNORECASE))
    flows = set(int(x) for x in re.findall(r"\bflow\s*sensor\s*([0-9]+)\b", s, flags=re.IGNORECASE))

    return tanks, pumps, valves, flows

def infer_subsystems(type_text, effect_phys):
    """
    Correct rule for Set 1 attack labeling:
    - Cyber attacks map via PLC mentioned in Type of attack (even if no physical effect)
    - Physical faults map via assets in Effect on physical process
    - Also allow asset mapping for any row because it can mention valve/tank directly
    """
    affected = set()

    # PLC mapping from Type of attack
    plcs = extract_plcs(type_text)
    for plc in plcs:
        affected.add(plc)  # PLC1->S1, PLC2->S2, PLC3->S3, PLC4->S4

    # Asset mapping from effect text
    tanks, pumps, valves, flows = extract_assets(effect_phys)

    if (tanks & S1["tanks"]) or (pumps & S1["pumps"]) or (valves & S1["valves"]) or (flows & S1["flows"]):
        affected.add(1)
    if (tanks & S2["tanks"]) or (pumps & S2["pumps"]) or (valves & S2["valves"]) or (flows & S2["flows"]):
        affected.add(2)

    return affected

def label_from_windows(times, windows_df, subsystem_id):
    labels = pd.Series(["Normal"] * len(times), index=times.index)

    w = windows_df[windows_df["subsystems"].apply(lambda s: subsystem_id in s)].copy()
    if w.empty:
        return labels

    # precedence
    for atk in ["MiTM", "DoS", "Physical"]:
        ww = w[w["attack_state"] == atk]
        for _, row in ww.iterrows():
            mask = (times >= row["start"]) & (times <= row["end"])
            labels.loc[mask] = atk

    return labels

def windows_from_readme_sheet(readme_path, sheet_name):
    df = pd.ExcelFile(readme_path).parse(sheet_name)

    start_col = _find_col(df, ["Start time", "Start"])
    end_col   = _find_col(df, ["End time", "End"])
    type_col  = _find_col(df, ["Type of attack", "Attack"])
    eff_col   = _find_col(df, ["Effect on physical process", "Effect on physical"])

    out = df[[start_col, end_col, type_col, eff_col]].copy()
    out.columns = ["start", "end", "type_text", "effect_phys"]

    out["start"] = pd.to_datetime(out["start"], errors="coerce", dayfirst=True)
    out["end"]   = pd.to_datetime(out["end"], errors="coerce", dayfirst=True)
    out = out.dropna(subset=["start", "end"]).copy()

    out["attack_state"] = out["type_text"].apply(parse_attack_state)
    out["subsystems"] = out.apply(lambda r: infer_subsystems(r["type_text"], r["effect_phys"]), axis=1)

    # keep rows that map to at least one subsystem
    out = out[out["subsystems"].apply(lambda s: len(s) > 0)].copy()

    return out

def counts_from_labels(labels, states=attack_states):
    c = labels.value_counts().to_dict()
    return {s: int(c.get(s, 0)) for s in states}

def add_counts(a, b):
    return {k: a.get(k, 0) + b.get(k, 0) for k in attack_states}

def counts_to_prob(counts):
    total = sum(counts.values())
    p = np.array([counts[s] for s in attack_states], dtype=float)
    if total > 0:
        p = p / total
    return p

def compute_attack_prior_across_acquisitions(csv_paths, readme_sheets, subsystem_id=1):
    if len(csv_paths) != len(readme_sheets):
        raise ValueError("csv_paths and readme_sheets must have the same length, one sheet per csv.")

    total_counts = {s: 0 for s in attack_states}
    total_seconds = 0

    per_acq = []

    for csv_path, sheet in zip(csv_paths, readme_sheets):
        df = read_csv_robust(csv_path)
        time_col = _find_col(df, ["Time", "Timestamp", "DateTime"])
        df[time_col] = pd.to_datetime(df[time_col], errors="coerce", dayfirst=True)
        df = df.dropna(subset=[time_col]).sort_values(time_col).reset_index(drop=True)
        times = df[time_col]

        t0, t1 = times.iloc[0], times.iloc[-1]

        win = windows_from_readme_sheet(README_XLSX_PATH, sheet)
        # restrict to this csv span so "Normal" is counted correctly
        win = win[(win["end"] >= t0) & (win["start"] <= t1)].copy()

        labels = label_from_windows(times, win, subsystem_id=subsystem_id)
        c = counts_from_labels(labels)

        total_counts = add_counts(total_counts, c)
        total_seconds += len(labels)

        per_acq.append((sheet, t0, t1, len(labels), c))

    p = counts_to_prob(total_counts)
    return p, total_counts, total_seconds, per_acq

# -----------------------------
# USE IT
# -----------------------------
csv_paths = [
    "phy_att_1.csv",
    "phy_att_2.csv",
    "phy_att_3.csv",
    "phy_att_4.csv",
]

readme_sheets = ["attack_1", "attack_2", "attack_3", "attack_4"]

p_attack1_all, counts_all, total_seconds, per_acq = compute_attack_prior_across_acquisitions(
    csv_paths, readme_sheets, subsystem_id=1
)

print("TOTAL seconds across acquisitions:", total_seconds)
print("TOTAL counts S1:", counts_all)
print("P(AttackType_1) across all acquisitions [Normal, MiTM, DoS, Physical]:", p_attack1_all)

print("\nPer acquisition breakdown")
for sheet, t0, t1, nsec, c in per_acq:
    print(sheet, "|", t0, "to", t1, "| seconds", nsec, "| counts", c)


TOTAL seconds across acquisitions: 7495
TOTAL counts S1: {'Normal': 6574, 'MiTM': 91, 'DoS': 280, 'Physical': 550}
P(AttackType_1) across all acquisitions [Normal, MiTM, DoS, Physical]: [0.87711808 0.01214143 0.03735824 0.07338225]

Per acquisition breakdown
attack_1 | 2021-04-09 18:23:28 to 2021-04-09 19:03:47 | seconds 2420 | counts {'Normal': 2055, 'MiTM': 88, 'DoS': 0, 'Physical': 277}
attack_2 | 2021-04-19 15:37:12 to 2021-04-19 16:12:16 | seconds 2104 | counts {'Normal': 1980, 'MiTM': 2, 'DoS': 80, 'Physical': 42}
attack_3 | 2021-04-09 19:42:12 to 2021-04-09 20:03:05 | seconds 1254 | counts {'Normal': 1109, 'MiTM': 0, 'DoS': 47, 'Physical': 98}
attack_4 | 2022-02-21 14:45:18 to 2022-02-21 15:13:54 | seconds 1717 | counts {'Normal': 1430, 'MiTM': 1, 'DoS': 153, 'Physical': 133}


### P(Attack Type 2 | Attack Type 1)

In [51]:
import pandas as pd
import numpy as np
import re

# README_XLSX_PATH = "/mnt/data/README.xlsx"

attack_states = ["Normal", "MiTM", "DoS", "Physical"]
state_to_idx = {s:i for i,s in enumerate(attack_states)}

# ----------------------------
# Subsystem definitions you gave
# ----------------------------
S1 = {
    "tanks": set(range(1, 6)),      # 1..5
    "pumps": set(range(1, 5)),      # 1..4
    "valves": set(range(1, 20)),    # 1..19
    "flows": set()
}
S2 = {
    "tanks": {6},
    "pumps": set(),
    "valves": {20},
    "flows": {1}                   # Flow sensor 1
}

# ----------------------------
# Helpers
# ----------------------------
def _find_col(df, candidates):
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    for c in df.columns:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    raise KeyError(f"Could not find columns like {candidates}. Found: {list(df.columns)}")

def parse_attack_state(type_text):
    s = str(type_text).lower()

    if "physical fault" in s or "breakdown" in s or "water leak" in s or "leak" in s:
        return "Physical"

    if "flood" in s or "dos" in s or "denial" in s or "icmp" in s or "tcp flood" in s or "udp flood" in s:
        return "DoS"

    if "mitm" in s or "man in the middle" in s or "scan" in s or "syn scan" in s or "fin scan" in s or "xmas" in s or "null scan" in s:
        return "MiTM"

    return "MiTM"

def extract_plcs(type_text):
    s = str(type_text)
    hits = re.findall(r"PLC\s*([1-4])", s, flags=re.IGNORECASE)
    return set(int(h) for h in hits)

def extract_assets(effect_text):
    s = str(effect_text)

    tanks = set(int(x) for x in re.findall(r"\btank\s*([0-9]+)\b", s, flags=re.IGNORECASE))
    pumps = set(int(x) for x in re.findall(r"\bpump\s*([0-9]+)\b", s, flags=re.IGNORECASE))
    valves = set(int(x) for x in re.findall(r"\bvalve\s*([0-9]+)\b", s, flags=re.IGNORECASE))
    flows = set(int(x) for x in re.findall(r"\bflow\s*sensor\s*([0-9]+)\b", s, flags=re.IGNORECASE))

    return tanks, pumps, valves, flows

def infer_subsystems(type_text, effect_phys):
    """
    Correct mapping
    - cyber -> PLC mapping from Type of attack
    - physical faults -> assets from Effect on physical process
    - also allow asset mapping always
    """
    affected = set()

    # PLC mapping
    plcs = extract_plcs(type_text)
    for plc in plcs:
        affected.add(plc)  # PLC1->S1 etc

    # asset mapping
    tanks, pumps, valves, flows = extract_assets(effect_phys)

    if (tanks & S1["tanks"]) or (pumps & S1["pumps"]) or (valves & S1["valves"]) or (flows & S1["flows"]):
        affected.add(1)
    if (tanks & S2["tanks"]) or (pumps & S2["pumps"]) or (valves & S2["valves"]) or (flows & S2["flows"]):
        affected.add(2)

    return affected

def windows_from_readme_sheet(readme_path, sheet_name):
    df = pd.ExcelFile(readme_path).parse(sheet_name)

    start_col = _find_col(df, ["Start time", "Start"])
    end_col   = _find_col(df, ["End time", "End"])
    type_col  = _find_col(df, ["Type of attack", "Attack"])
    eff_col   = _find_col(df, ["Effect on physical process", "Effect on physical"])

    out = df[[start_col, end_col, type_col, eff_col]].copy()
    out.columns = ["start", "end", "type_text", "effect_phys"]

    out["start"] = pd.to_datetime(out["start"], errors="coerce", dayfirst=True)
    out["end"]   = pd.to_datetime(out["end"], errors="coerce", dayfirst=True)
    out = out.dropna(subset=["start", "end"]).copy()
    out = out[out["end"] >= out["start"]].copy()

    out["attack_state"] = out["type_text"].apply(parse_attack_state)
    out["subsystems"] = out.apply(lambda r: infer_subsystems(r["type_text"], r["effect_phys"]), axis=1)

    # keep rows that map to at least one subsystem
    out = out[out["subsystems"].apply(lambda s: len(s) > 0)].copy()

    return out

def build_attack_series(times, windows_df, subsystem_id):
    labels = pd.Series(["Normal"] * len(times), index=times.index)

    w = windows_df[windows_df["subsystems"].apply(lambda s: subsystem_id in s)].copy()
    if w.empty:
        return labels

    for atk in ["MiTM", "DoS", "Physical"]:  # precedence
        ww = w[w["attack_state"] == atk]
        for _, row in ww.iterrows():
            mask = (times >= row["start"]) & (times <= row["end"])
            labels.loc[mask] = atk

    return labels

def update_counts_joint(counts, child_labels, parent_labels):
    for c, p in zip(child_labels, parent_labels):
        counts[state_to_idx[c], state_to_idx[p]] += 1
    return counts

def smooth_and_normalize(counts, alpha=0.5):
    counts_sm = counts.astype(float) + alpha
    col_sums = counts_sm.sum(axis=0, keepdims=True)
    return counts_sm / col_sums

# ----------------------------
# MAIN: Set 2 across 1..4
# ----------------------------
# csv_paths = [
#     "/mnt/data/phy_att_1.csv",
#     "/mnt/data/phy_att_2.csv",
#     "/mnt/data/phy_att_3.csv",
#     "/mnt/data/phy_att_4.csv",
# ]
readme_sheets = ["attack_1", "attack_2", "attack_3", "attack_4"]

counts_21_total = np.zeros((4, 4), dtype=int)

per_acq_summ = []

for csv_path, sheet in zip(csv_paths, readme_sheets):
    df = read_csv_robust(csv_path)

    time_col = _find_col(df, ["Time", "Timestamp", "DateTime"])
    df[time_col] = pd.to_datetime(df[time_col], errors="coerce", dayfirst=True)
    df = df.dropna(subset=[time_col]).sort_values(time_col).reset_index(drop=True)
    times = df[time_col]

    t0, t1 = times.iloc[0], times.iloc[-1]

    win = windows_from_readme_sheet(README_XLSX_PATH, sheet)
    win = win[(win["end"] >= t0) & (win["start"] <= t1)].copy()

    A1 = build_attack_series(times, win, subsystem_id=1)
    A2 = build_attack_series(times, win, subsystem_id=2)

    counts_21_total = update_counts_joint(counts_21_total, A2.values, A1.values)

    per_acq_summ.append({
        "sheet": sheet,
        "csv": csv_path,
        "t0": t0, "t1": t1,
        "counts_A1": A1.value_counts().to_dict(),
        "counts_A2": A2.value_counts().to_dict(),
        "n": len(times)
    })

alpha = 0.5
cpt_21 = smooth_and_normalize(counts_21_total, alpha=alpha)

print("Raw joint counts counts[A2, A1] aggregated across all acquisitions")
print("Rows:", attack_states)
print("Cols:", attack_states)
print(counts_21_total)

cpt_df = pd.DataFrame(
    cpt_21,
    index=[f"A2={s}" for s in attack_states],
    columns=[f"A1={s}" for s in attack_states],
)
print(f"\nCPT P(AttackType_2 | AttackType_1) with alpha={alpha}")
print(cpt_df.round(4))

print("\nPer acquisition quick summary")
for r in per_acq_summ:
    print(r["sheet"], "| n", r["n"], "| A1", r["counts_A1"], "| A2", r["counts_A2"])


Raw joint counts counts[A2, A1] aggregated across all acquisitions
Rows: ['Normal', 'MiTM', 'DoS', 'Physical']
Cols: ['Normal', 'MiTM', 'DoS', 'Physical']
[[6287    3   35  550]
 [ 151   88    0    0]
 [   0    0  245    0]
 [ 136    0    0    0]]

CPT P(AttackType_2 | AttackType_1) with alpha=0.5
             A1=Normal  A1=MiTM  A1=DoS  A1=Physical
A2=Normal       0.9561   0.0376  0.1259       0.9973
A2=MiTM         0.0230   0.9516  0.0018       0.0009
A2=DoS          0.0001   0.0054  0.8706       0.0009
A2=Physical     0.0208   0.0054  0.0018       0.0009

Per acquisition quick summary
attack_1 | n 2420 | A1 {'Normal': 2055, 'Physical': 277, 'MiTM': 88} | A2 {'Normal': 2185, 'MiTM': 235}
attack_2 | n 2104 | A1 {'Normal': 1980, 'DoS': 80, 'Physical': 42, 'MiTM': 2} | A2 {'Normal': 1956, 'Physical': 81, 'DoS': 66, 'MiTM': 1}
attack_3 | n 1254 | A1 {'Normal': 1109, 'Physical': 98, 'DoS': 47} | A2 {'Normal': 1152, 'Physical': 55, 'DoS': 47}
attack_4 | n 1717 | A1 {'Normal': 1430, 'DoS': 

### P(Sensor|Attack Type)

In [52]:
import pandas as pd
import numpy as np
import re

# ------------------------------------------------------------
# States
# ------------------------------------------------------------
ATTACK_STATES = ["Normal", "MiTM", "DoS", "Physical"]
SENSOR_STATES = ["Normal", "Abnormal"]
a2i = {s: i for i, s in enumerate(ATTACK_STATES)}

# ------------------------------------------------------------
# Robust CSV reader (handles weird encodings + tab-separated files)
# ------------------------------------------------------------
# def read_csv_robust(path):
#     # Try common encodings, and handle "one giant tab-separated column" cases
#     encodings = ["utf-8", "utf-8-sig", "latin1", "utf-16", "utf-16-le", "cp1252"]
#     last_err = None

#     for enc in encodings:
#         try:
#             df = pd.read_csv(path, encoding=enc)
#             # If it came in as a single column with embedded tabs, split it
#             if df.shape[1] == 1 and ("\t" in df.columns[0]):
#                 df = pd.read_csv(path, encoding=enc, sep="\t")
#             return df
#         except Exception as e:
#             last_err = e
#             continue

#     # final fallback: python engine with auto-sep inference
#     try:
#         df = pd.read_csv(path, engine="python", sep=None)
#         if df.shape[1] == 1 and ("\t" in df.columns[0]):
#             df = pd.read_csv(path, engine="python", sep="\t")
#         return df
#     except Exception as e:
#         raise RuntimeError(f"Could not read CSV {path}. Last error: {last_err}") from e


def find_time_col(df):
    # Prefer exact "Time" but fall back to any column containing time/date
    for c in df.columns:
        if c.strip().lower() == "time":
            return c
    for c in df.columns:
        cl = c.lower()
        if "time" in cl or "date" in cl:
            return c
    raise KeyError(f"Could not find a time column. Columns are: {list(df.columns)}")


# ------------------------------------------------------------
# README parsing helpers
# ------------------------------------------------------------
def parse_attack_state(type_text):
    s = str(type_text).lower()

    # Physical
    if "physical fault" in s or "breakdown" in s or "water leak" in s:
        return "Physical"

    # DoS
    if "flood" in s or "dos" in s or "denial" in s:
        return "DoS"

    # MiTM bucket (includes scans)
    if "mitm" in s or "man in the middle" in s or "scan" in s:
        return "MiTM"

    return "Normal"


def extract_plcs(text):
    hits = re.findall(r"PLC\s*([1-4])", str(text), flags=re.IGNORECASE)
    return set(int(h) for h in hits)


def extract_assets(text):
    """
    Extract asset ids from free text.
    Works on both 'Type of attack' and 'Effect on physical process'.
    """
    s = str(text).lower()

    tanks = set(int(x) for x in re.findall(r"tank\s*(\d+)", s))
    pumps = set(int(x) for x in re.findall(r"pump\s*(\d+)", s))
    valves = set(int(x) for x in re.findall(r"valve\s*(\d+)", s))

    # flow sensors
    flows = set(int(x) for x in re.findall(r"flow\s*sensor\s*(\d+)", s))
    # some text writes "flow sensor 1 and PLC3"
    flows |= set(int(x) for x in re.findall(r"flow\s*sensor\s*([0-9]+)", s))

    return tanks, pumps, valves, flows


def load_attack_sheet(readme_xlsx, sheet_name):
    df = pd.read_excel(readme_xlsx, sheet_name=sheet_name)

    # Find required columns by fuzzy match
    cols = {c.lower(): c for c in df.columns}

    def pick(patterns):
        for p in patterns:
            for c in df.columns:
                if p in c.lower():
                    return c
        raise KeyError(f"Could not find column with patterns {patterns}. Columns: {list(df.columns)}")

    start_col = pick(["start"])
    end_col = pick(["end"])
    type_col = pick(["type of attack", "attack type", "type"])
    effect_col = pick(["effect on physical process", "effect on physical"])

    out = df[[start_col, end_col, type_col, effect_col]].copy()
    out.columns = ["start", "end", "type_text", "effect_phys"]

    out["start"] = pd.to_datetime(out["start"], errors="coerce", dayfirst=True)
    out["end"] = pd.to_datetime(out["end"], errors="coerce", dayfirst=True)
    out = out.dropna(subset=["start", "end"])
    out = out[out["end"] >= out["start"]].copy()

    # Parse attack state
    out["attack_state"] = out["type_text"].apply(parse_attack_state)

    # Parse PLCs and assets (use both columns to catch physical faults)
    plcs_list = []
    tanks_list, pumps_list, valves_list, flows_list = [], [], [], []
    for _, r in out.iterrows():
        plcs = extract_plcs(r["type_text"])
        t1, p1, v1, f1 = extract_assets(r["type_text"])
        t2, p2, v2, f2 = extract_assets(r["effect_phys"])

        plcs_list.append(plcs)
        tanks_list.append(t1 | t2)
        pumps_list.append(p1 | p2)
        valves_list.append(v1 | v2)
        flows_list.append(f1 | f2)

    out["plcs"] = plcs_list
    out["tanks"] = tanks_list
    out["pumps"] = pumps_list
    out["valves"] = valves_list
    out["flows"] = flows_list

    return out


# ------------------------------------------------------------
# Subsystem definitions (your stated mapping)
# ------------------------------------------------------------
SUBSYSTEMS = {
    1: {
        "tanks": set(range(1, 6)),         # 1-5
        "pumps": set(range(1, 5)),         # 1-4
        "valves": set(range(1, 20)),       # 1-19
        "flows": set(),                    # none in S1
    },
    2: {
        "tanks": {6},                      # tank 6
        "pumps": set(),                    # none in S2
        "valves": {20},                    # valve 20
        "flows": {1},                      # flow sensor 1
    }
}


# ------------------------------------------------------------
# Build time-expanded labels for AttackType_s(t) and SensorState_s(t)
# ------------------------------------------------------------
def build_labels_for_subsystem_and_sensor(times, events_df, subsystem_id, sensor_type):
    """
    sensor_type in {"tank","pump","valve","flow"}
    AttackType_s(t): determined by any event affecting subsystem s
    SensorState_s(t): Abnormal only if event affects subsystem s AND touches that sensor type
    Precedence for attack overlap: Physical > DoS > MiTM > Normal
    """
    assets = SUBSYSTEMS[subsystem_id]
    n = len(times)

    attack_labels = pd.Series(["Normal"] * n)
    sensor_labels = pd.Series(["Normal"] * n)

    # If this sensor type does not exist in this subsystem, it stays Normal always
    if len(assets[sensor_type + "s" if sensor_type != "flow" else "flows"]) == 0:
        return attack_labels, sensor_labels

    # Filter events that overlap this CSV time range
    csv_start, csv_end = times.iloc[0], times.iloc[-1]
    e = events_df[(events_df["start"] <= csv_end) & (events_df["end"] >= csv_start)].copy()
    if e.empty:
        return attack_labels, sensor_labels

    def subsystem_affected(row):
        atk = row["attack_state"]

        # cyber -> PLC mapping: PLC1->S1, PLC2->S2, PLC3->S3, PLC4->S4
        if atk in ["MiTM", "DoS"]:
            return subsystem_id in row["plcs"]

        # physical -> asset mapping using your subsystem assets
        if atk == "Physical":
            if (row["tanks"] & assets["tanks"]) or (row["pumps"] & assets["pumps"]) or (row["valves"] & assets["valves"]) or (row["flows"] & assets["flows"]):
                return True
        return False

    def sensor_affected(row):
        # Only for rows that already affect subsystem
        if sensor_type == "tank":
            return len(row["tanks"] & assets["tanks"]) > 0
        if sensor_type == "pump":
            return len(row["pumps"] & assets["pumps"]) > 0
        if sensor_type == "valve":
            return len(row["valves"] & assets["valves"]) > 0
        if sensor_type == "flow":
            return len(row["flows"] & assets["flows"]) > 0
        return False

    # Apply in increasing precedence so later overwrites earlier
    for atk in ["MiTM", "DoS", "Physical"]:
        ee = e[e["attack_state"] == atk]
        for _, row in ee.iterrows():
            if not subsystem_affected(row):
                continue

            mask = (times >= row["start"]) & (times <= row["end"])
            attack_labels.loc[mask] = atk

            if sensor_affected(row):
                sensor_labels.loc[mask] = "Abnormal"

    return attack_labels, sensor_labels


# ------------------------------------------------------------
# Emission CPT aggregation across acquisitions
# ------------------------------------------------------------
def aggregate_emission_cpt_across_acquisitions(
    csv_paths,
    readme_xlsx,
    readme_sheets,
    subsystem_id,
    sensor_type,         # "tank"|"valve"|"pump"|"flow"
    alpha=0.5
):
    """
    Returns:
      counts_total: shape [2,4] (Normal/Abnormal x AttackType)
      cpt:          shape [2,4] columns sum to 1
      per_file:     list of dict summaries
    """
    counts_total = np.zeros((2, 4), dtype=int)
    per_file = []

    for csv_path, sheet in zip(csv_paths, readme_sheets):
        df = read_csv_robust(csv_path)
        tcol = find_time_col(df)
        df[tcol] = pd.to_datetime(df[tcol], errors="coerce", dayfirst=True)
        df = df.dropna(subset=[tcol]).sort_values(tcol).reset_index(drop=True)
        times = df[tcol]

        events = load_attack_sheet(readme_xlsx, sheet)

        attack_labels, sensor_labels = build_labels_for_subsystem_and_sensor(
            times=times,
            events_df=events,
            subsystem_id=subsystem_id,
            sensor_type=sensor_type
        )

        # Count over CSV time points
        counts = np.zeros((2, 4), dtype=int)
        for s, a in zip(sensor_labels.values, attack_labels.values):
            si = 0 if s == "Normal" else 1
            ai = a2i[a]
            counts[si, ai] += 1

        counts_total += counts

        per_file.append({
            "csv": csv_path,
            "sheet": sheet,
            "time_start": times.iloc[0],
            "time_end": times.iloc[-1],
            "attack_counts": pd.Series(attack_labels.values).value_counts().to_dict(),
            "sensor_counts": pd.Series(sensor_labels.values).value_counts().to_dict(),
            "counts_matrix": counts
        })

    # Smooth + normalize per AttackType column
    counts_sm = counts_total.astype(float) + alpha
    cpt = counts_sm / counts_sm.sum(axis=0, keepdims=True)

    return counts_total, cpt, per_file


# ------------------------------------------------------------
# Convenience: compute ALL emissions for both subsystems and all sensor nodes
# ------------------------------------------------------------
def compute_all_emissions(csv_paths, readme_xlsx, readme_sheets, alpha=0.5):
    results = {}
    for subsystem_id in [1, 2]:
        for sensor_type in ["tank", "valve", "pump", "flow"]:
            key = f"S{subsystem_id}_{sensor_type}"
            counts, cpt, per_file = aggregate_emission_cpt_across_acquisitions(
                csv_paths=csv_paths,
                readme_xlsx=readme_xlsx,
                readme_sheets=readme_sheets,
                subsystem_id=subsystem_id,
                sensor_type=sensor_type,
                alpha=alpha
            )
            results[key] = {
                "counts_total": counts,
                "cpt": cpt,
                "per_file": per_file
            }
    return results


# ------------------------------------------------------------
# Example usage (edit paths to your local files)
# ------------------------------------------------------------
if __name__ == "__main__":
    csv_paths = ["phy_att_1.csv", "phy_att_2.csv", "phy_att_3.csv", "phy_att_4.csv"]
    readme_sheets = ["attack_1", "attack_2", "attack_3", "attack_4"]
    readme_xlsx = "README.xlsx"

    results = compute_all_emissions(csv_paths, readme_xlsx, readme_sheets, alpha=0.5)

    # Print one example: Subsystem 2, flow sensor
    k = "S2_flow"
    print("\n", k)
    print("Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:\n", results[k]["counts_total"])
    print("\nCPT P(SensorState | AttackType):")
    print(pd.DataFrame(results[k]["cpt"], index=SENSOR_STATES, columns=ATTACK_STATES).round(4))



 S2_flow
Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:
 [[7091  239   29  136]
 [   0    0    0    0]]

CPT P(SensorState | AttackType):
          Normal    MiTM     DoS  Physical
Normal    0.9999  0.9979  0.9833    0.9964
Abnormal  0.0001  0.0021  0.0167    0.0036


In [57]:
k = "S2_pump"
print("\n", k)
print("Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:\n", results[k]["counts_total"])
print("\nCPT P(SensorState | AttackType):")
print(pd.DataFrame(results[k]["cpt"], index=SENSOR_STATES, columns=ATTACK_STATES).round(4))


 S2_pump
Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:
 [[7495    0    0    0]
 [   0    0    0    0]]

CPT P(SensorState | AttackType):
          Normal  MiTM  DoS  Physical
Normal    0.9999   0.5  0.5       0.5
Abnormal  0.0001   0.5  0.5       0.5


In [54]:
k = "S2_valve"
print("\n", k)
print("Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:\n", results[k]["counts_total"])
print("\nCPT P(SensorState | AttackType):")
print(pd.DataFrame(results[k]["cpt"], index=SENSOR_STATES, columns=ATTACK_STATES).round(4))


 S2_valve
Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:
 [[7091  151    0    0]
 [   0   88   29  136]]

CPT P(SensorState | AttackType):
          Normal    MiTM     DoS  Physical
Normal    0.9999  0.6312  0.0167    0.0036
Abnormal  0.0001  0.3688  0.9833    0.9964


In [55]:
k = "S2_tank"
print("\n", k)
print("Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:\n", results[k]["counts_total"])
print("\nCPT P(SensorState | AttackType):")
print(pd.DataFrame(results[k]["cpt"], index=SENSOR_STATES, columns=ATTACK_STATES).round(4))


 S2_tank
Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:
 [[7091  151    0    0]
 [   0   88   29  136]]

CPT P(SensorState | AttackType):
          Normal    MiTM     DoS  Physical
Normal    0.9999  0.6312  0.0167    0.0036
Abnormal  0.0001  0.3688  0.9833    0.9964


In [56]:
k = "S1_valve"
print("\n", k)
print("Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:\n", results[k]["counts_total"])
print("\nCPT P(SensorState | AttackType):")
print(pd.DataFrame(results[k]["cpt"], index=SENSOR_STATES, columns=ATTACK_STATES).round(4))


 S1_valve
Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:
 [[6574   91  280  550]
 [   0    0    0    0]]

CPT P(SensorState | AttackType):
          Normal    MiTM     DoS  Physical
Normal    0.9999  0.9946  0.9982    0.9991
Abnormal  0.0001  0.0054  0.0018    0.0009


In [58]:
k = "S1_tank"
print("\n", k)
print("Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:\n", results[k]["counts_total"])
print("\nCPT P(SensorState | AttackType):")
print(pd.DataFrame(results[k]["cpt"], index=SENSOR_STATES, columns=ATTACK_STATES).round(4))


 S1_tank
Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:
 [[6574   91  280    0]
 [   0    0    0  550]]

CPT P(SensorState | AttackType):
          Normal    MiTM     DoS  Physical
Normal    0.9999  0.9946  0.9982    0.0009
Abnormal  0.0001  0.0054  0.0018    0.9991


In [59]:
k = "S1_pump"
print("\n", k)
print("Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:\n", results[k]["counts_total"])
print("\nCPT P(SensorState | AttackType):")
print(pd.DataFrame(results[k]["cpt"], index=SENSOR_STATES, columns=ATTACK_STATES).round(4))


 S1_pump
Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:
 [[6574   91  280  260]
 [   0    0    0  290]]

CPT P(SensorState | AttackType):
          Normal    MiTM     DoS  Physical
Normal    0.9999  0.9946  0.9982    0.4728
Abnormal  0.0001  0.0054  0.0018    0.5272


In [60]:
k = "S1_flow"
print("\n", k)
print("Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:\n", results[k]["counts_total"])
print("\nCPT P(SensorState | AttackType):")
print(pd.DataFrame(results[k]["cpt"], index=SENSOR_STATES, columns=ATTACK_STATES).round(4))


 S1_flow
Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:
 [[7495    0    0    0]
 [   0    0    0    0]]

CPT P(SensorState | AttackType):
          Normal  MiTM  DoS  Physical
Normal    0.9999   0.5  0.5       0.5
Abnormal  0.0001   0.5  0.5       0.5


In [63]:
import numpy as np
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

np.random.seed(42)

attack_order = ["Normal", "MiTM", "DoS", "Physical"]
binary_order = ["Normal", "Abnormal"]

P_A1 = np.array([0.87711808, 0.01214143, 0.03735824, 0.07338225], dtype=float)

A21 = np.array([
    [0.9561, 0.0376, 0.1259, 0.9973],
    [0.0230, 0.9516, 0.0018, 0.0009],
    [0.0001, 0.0054, 0.8705, 0.0009],
    [0.0208, 0.0054, 0.0018, 0.0009],
], dtype=float)

def make_emission_cpt_2x4(p_abnormal_by_attack_4, eps=1e-6):
    p_ab = np.array(p_abnormal_by_attack_4, dtype=float).reshape(4)
    p_ab = np.clip(p_ab, eps, 1.0 - eps)
    cpt = np.zeros((2, 4), dtype=float)
    cpt[1, :] = p_ab
    cpt[0, :] = 1.0 - p_ab
    if not np.allclose(cpt.sum(axis=0), 1.0):
        raise ValueError("Emission CPT columns must sum to 1.")
    return cpt

# emissions
E_valve1 = make_emission_cpt_2x4([0.00, 0.01, 0.05, 0.99])
E_pump1  = make_emission_cpt_2x4([0.00, 0.05, 0.10, 0.55])
E_tank1  = make_emission_cpt_2x4([0.00, 0.01, 0.05, 0.99])

E_valve2 = make_emission_cpt_2x4([0.00, 0.01, 0.05, 0.99])
E_flow2  = make_emission_cpt_2x4([0.00, 0.05, 0.05, 0.45])
E_tank2  = make_emission_cpt_2x4([0.00, 0.01, 0.05, 0.99])

# BN structure
model = DiscreteBayesianNetwork([
    ("AttackType_1", "AttackType_2"),
    ("AttackType_1", "ValveState_1"),
    ("AttackType_1", "PumpState_1"),
    ("AttackType_1", "TankState_1"),
    ("AttackType_2", "ValveState_2"),
    ("AttackType_2", "FlowState_2"),
    ("AttackType_2", "TankState_2"),
])

# CPDs (all values are shaped (child_card, prod(parent_cards)))
cpd_A1 = TabularCPD(
    "AttackType_1", 4, values=P_A1.reshape(4, 1),
    state_names={"AttackType_1": attack_order}
)

cpd_A2 = TabularCPD(
    "AttackType_2", 4, values=A21,
    evidence=["AttackType_1"], evidence_card=[4],
    state_names={"AttackType_2": attack_order, "AttackType_1": attack_order}
)

cpd_V1 = TabularCPD(
    "ValveState_1", 2, values=E_valve1,
    evidence=["AttackType_1"], evidence_card=[4],
    state_names={"ValveState_1": binary_order, "AttackType_1": attack_order}
)
cpd_P1 = TabularCPD(
    "PumpState_1", 2, values=E_pump1,
    evidence=["AttackType_1"], evidence_card=[4],
    state_names={"PumpState_1": binary_order, "AttackType_1": attack_order}
)
cpd_T1 = TabularCPD(
    "TankState_1", 2, values=E_tank1,
    evidence=["AttackType_1"], evidence_card=[4],
    state_names={"TankState_1": binary_order, "AttackType_1": attack_order}
)

cpd_V2 = TabularCPD(
    "ValveState_2", 2, values=E_valve2,
    evidence=["AttackType_2"], evidence_card=[4],
    state_names={"ValveState_2": binary_order, "AttackType_2": attack_order}
)
cpd_F2 = TabularCPD(
    "FlowState_2", 2, values=E_flow2,
    evidence=["AttackType_2"], evidence_card=[4],
    state_names={"FlowState_2": binary_order, "AttackType_2": attack_order}
)
cpd_T2 = TabularCPD(
    "TankState_2", 2, values=E_tank2,
    evidence=["AttackType_2"], evidence_card=[4],
    state_names={"TankState_2": binary_order, "AttackType_2": attack_order}
)

model.add_cpds(cpd_A1, cpd_A2, cpd_V1, cpd_P1, cpd_T1, cpd_V2, cpd_F2, cpd_T2)

assert model.check_model(), "Invalid BN (check CPTs and structure)."

# exact inference
infer = VariableElimination(model)

evidence = {"FlowState_2": "Abnormal", "ValveState_2": "Abnormal"}
posterior = infer.query(["AttackType_2"], evidence=evidence, show_progress=False)
states = posterior.state_names["AttackType_2"]
vals = posterior.values

for s, v in zip(states, vals):
    print(f"P(AttackType_2={s} | evidence) = {float(v):.6f}")



P(AttackType_2=Normal | evidence) = 0.000000
P(AttackType_2=MiTM | evidence) = 0.001916
P(AttackType_2=DoS | evidence) = 0.009845
P(AttackType_2=Physical | evidence) = 0.988239


In [65]:
evidence = {"FlowState_2": "Abnormal", "ValveState_2": "Abnormal"}
posterior = infer.query(["AttackType_1"], evidence=evidence, show_progress=False)
states = posterior.state_names["AttackType_1"]
vals = posterior.values

for s, v in zip(states, vals):
    print(f"P(AttackType_1={s} | evidence) = {float(v):.6f}")

P(AttackType_1=Normal | evidence) = 0.978824
P(AttackType_1=MiTM | evidence) = 0.004228
P(AttackType_1=DoS | evidence) = 0.013386
P(AttackType_1=Physical | evidence) = 0.003563


In [67]:
evidence = {"AttackType_2": "MiTM"}
posterior = infer.query(["AttackType_1"], evidence=evidence, show_progress=False)
states = posterior.state_names["AttackType_1"]
vals = posterior.values

for s, v in zip(states, vals):
    print(f"P(AttackType_1={s} | evidence) = {float(v):.6f}")

P(AttackType_1=Normal | evidence) = 0.633183
P(AttackType_1=MiTM | evidence) = 0.362633
P(AttackType_1=DoS | evidence) = 0.002111
P(AttackType_1=Physical | evidence) = 0.002073
